# Notebook 5 — Sampling Frame

Joins monthly flood extent data to two administrative boundary sources and H3-7 hexagons.
Outputs GeoParquet and CSV tables suitable for phone-survey sampling frame design.

**Provinces:** North Kivu, South Kivu, Ituri  
**Period:** 2025-03 – 2026-02 (valid months only)  
**Admin-2:** World Bank geoBoundaries COD ADM2 (territories, ~41 in AOI)  
**Admin-3:** OCHA HDX COD-AB DRC (secteur / chefferie level)  
**Hex grid:** H3 resolution 7 (~5 km²)  
**Cell towers:** OpenCelliD DRC (MCC=630) — requires API token in `.env`


In [1]:
from pathlib import Path
import os

def _find_project_root():
    for candidate in [Path(os.path.abspath('')), Path(os.path.abspath('')).parent]:
        if (candidate / 'config' / 'config.yaml').exists():
            return candidate
    raise FileNotFoundError('Cannot locate config/config.yaml. Run from project root or notebooks/.')

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\trevm\Projects\Floodmaps


In [2]:
import importlib, subprocess, sys
needed = ['geopandas', 'pandas', 'shapely', 'h3', 'requests', 'pyarrow', 'tqdm']
missing = [p for p in needed if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
    print(f'Installed: {missing}')
else:
    print('All packages present.')

All packages present.


In [3]:
import json, warnings
import requests
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import shape, mapping, Polygon
from tqdm import tqdm
warnings.filterwarnings('ignore')

OUTPUT_DIR  = Path('data/outputs/flood_extent')
FRAMES_DIR  = Path('data/outputs/sampling_frames')
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

PROVINCES   = ['North Kivu', 'Sud-Kivu', 'Ituri']   # geoBoundaries ADM1 name variants
AOI_BBOX    = dict(min_lon=26.8, max_lon=30.8, min_lat=-5.9, max_lat=3.0)
H3_RES      = 7
BAD_MONTHS  = {'2025-01', '2025-02'}
GAP_MONTHS  = set()   # 2026-03/04 recovered via MPC RTC (2026-07-10);
                      # 2026-06 recovered 2026-09-08 — no month is a gap now

print('Config ready.')

Config ready.


## 1  Load flood data

In [4]:
df = pd.read_csv(OUTPUT_DIR / 'flood_stats.csv', index_col='month').sort_index()
df['quality'] = 'valid'
df.loc[df.index.isin(BAD_MONTHS), 'quality'] = 'bad'
df.loc[df.index.isin(GAP_MONTHS),  'quality'] = 'gap'
valid_months = df[df['quality'] == 'valid'].index.tolist()

flood_gdfs = {}
for month in valid_months:
    p = OUTPUT_DIR / f'flood_extent_{month}.geojson'
    if p.exists():
        flood_gdfs[month] = gpd.read_file(p)

print(f'Loaded {len(flood_gdfs)} valid flood layers: {list(flood_gdfs.keys())}')

Loaded 16 valid flood layers: ['2025-03', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']


## 2  World Bank admin-2 boundaries (geoBoundaries)

Downloads DRC ADM1 (provinces) and ADM2 (territories) from the geoBoundaries API.
ADM2 = territoire level — 41 territories intersect North Kivu, South Kivu, and Ituri.


In [5]:
def fetch_geoboundaries(iso3, level, cache_dir=Path('data/raw/boundaries')):
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f'{iso3}_{level}.geojson'
    if cache_file.exists():
        print(f'  Using cached {cache_file}')
        return gpd.read_file(cache_file)
    url = f'https://www.geoboundaries.org/api/current/gbOpen/{iso3}/{level}/'
    meta = requests.get(url, timeout=30).json()
    dl_url = meta['gjDownloadURL']
    print(f'  Downloading {iso3} {level} from geoBoundaries...')
    gdf = gpd.read_file(dl_url)
    gdf.to_file(cache_file, driver='GeoJSON')
    print(f'  Cached to {cache_file}')
    return gdf

print('Fetching ADM1 (provinces)...')
adm1 = fetch_geoboundaries('COD', 'ADM1')
print(f'  ADM1 columns: {list(adm1.columns)}')
print(adm1[['shapeName']].drop_duplicates().head(10).to_string())

Fetching ADM1 (provinces)...
  Using cached data\raw\boundaries\COD_ADM1.geojson


  ADM1 columns: ['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']
      shapeName
0    Upper Uele
1         Ituri
2        Tshopo
3    Lower Uele
4       Mongala
5   Nord-Ubangi
6       Tshuapa
7      Équateur
8  Haut-Katanga
9   Haut-Lomami


In [6]:
# Filter ADM1 to our three provinces (match on name containing key terms)
prov_mask = adm1['shapeName'].str.contains('Kivu|Ituri', case=False, na=False)
target_provs = adm1[prov_mask]
print('Target provinces found:')
print(target_provs[['shapeName']].to_string())

Target provinces found:
     shapeName
1        Ituri
18  North Kivu
19  South Kivu


In [7]:
print('Fetching ADM2 (territories)...')
adm2 = fetch_geoboundaries('COD', 'ADM2')
print(f'  Total ADM2 features: {len(adm2)}')

prov_union = target_provs.dissolve().geometry.iloc[0]
adm2_aoi = adm2[adm2.geometry.centroid.within(prov_union)].copy()
adm2_aoi = adm2_aoi.to_crs('EPSG:4326')
print(f'  ADM2 territories in AOI: {len(adm2_aoi)}')
print(adm2_aoi[['shapeName', 'shapeISO']].head(10).to_string())


Fetching ADM2 (territories)...
  Using cached data\raw\boundaries\COD_ADM2.geojson


  Total ADM2 features: 189

  ADM2 territories in AOI: 29
   shapeName shapeISO
3    Ariwara         
4        Aru         
11    Baraka         
16      Beni         
29    Bukavu         
32     Bunia         
35   Butembo         
44     Djugu         
48      Fizi         
51      Goma         


## 2b  OCHA HDX admin-3 boundaries (secteur / chefferie)

Downloads DRC ADM3 boundaries from the OCHA HDX Common Operational Dataset (COD-AB).  
For DRC, ADM3 = secteur, chefferie, or commune — the unit below territoire.  
Source: https://data.humdata.org/dataset/drc-administrative-boundaries


In [8]:
import io, zipfile, tempfile

def fetch_adm3_drc(cache_path=Path('data/raw/boundaries/drc_adm3.gpkg')):
    """Download DRC admin-3 from OCHA HDX COD-AB, falling back to geoBoundaries ADM3."""
    if cache_path.exists():
        print(f'  Loaded ADM3 from cache: {cache_path}')
        return gpd.read_file(str(cache_path))

    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # --- Attempt 1: OCHA HDX CKAN API ---
    api_base = 'https://data.humdata.org/api/3/action/package_show'
    for ds_id in ['drc-administrative-boundaries', 'cod-ab-drc',
                  'democratic-republic-of-the-congo-administrative-boundaries']:
        try:
            r = requests.get(api_base, params={'id': ds_id}, timeout=15)
            if r.ok and r.json().get('success'):
                resources = r.json()['result']['resources']
                adm3_res = next(
                    (res for res in resources
                     if 'adm3' in res.get('name', '').lower() or 'adm3' in res.get('url', '').lower()),
                    None
                )
                if adm3_res:
                    dl_url = adm3_res.get('download_url') or adm3_res.get('url')
                    print(f'  Downloading ADM3 from HDX: {dl_url}')
                    resp = requests.get(dl_url, stream=True, timeout=180, allow_redirects=True)
                    resp.raise_for_status()
                    if '.zip' in dl_url.lower() or 'zip' in resp.headers.get('content-type', ''):
                        with tempfile.TemporaryDirectory() as tmpdir:
                            zpath = Path(tmpdir) / 'adm3.zip'
                            with open(zpath, 'wb') as fout:
                                for chunk in resp.iter_content(8192):
                                    fout.write(chunk)
                            with zipfile.ZipFile(zpath) as zf:
                                zf.extractall(tmpdir)
                            shp = sorted(Path(tmpdir).rglob('*adm3*.shp')) or sorted(Path(tmpdir).rglob('*.shp'))
                            gdf = gpd.read_file(str(shp[0]))
                    else:
                        gdf = gpd.read_file(io.BytesIO(resp.content))
                    gdf = gdf.to_crs('EPSG:4326')
                    gdf.to_file(str(cache_path), driver='GPKG')
                    print(f'  Cached {len(gdf)} ADM3 features from HDX')
                    return gdf
        except Exception as e:
            print(f'  HDX attempt failed ({ds_id}): {e}')
            continue

    # --- Attempt 2: geoBoundaries ADM3 (same underlying data) ---
    print('  HDX unreachable — falling back to geoBoundaries ADM3...')
    gdf = fetch_geoboundaries('COD', 'ADM3')
    gdf = gdf.to_crs('EPSG:4326')
    gdf.to_file(str(cache_path), driver='GPKG')
    print(f'  Cached {len(gdf)} ADM3 features from geoBoundaries')
    return gdf


hdx_adm3 = fetch_adm3_drc()

# Filter to three target provinces using centroid containment
adm3_aoi = hdx_adm3[hdx_adm3.geometry.centroid.within(prov_union)].copy().to_crs('EPSG:4326')
print(f'  ADM3 units in AOI: {len(adm3_aoi)}')

# Identify name column (varies by source)
name_candidates = [c for c in adm3_aoi.columns
                   if ('adm3' in c.lower() or 'name' in c.lower()) and 'pcode' not in c.lower()]
adm3_name_col = name_candidates[0] if name_candidates else adm3_aoi.columns[0]
print(f'  Name column: {adm3_name_col}')
adm3_aoi = adm3_aoi.rename(columns={adm3_name_col: 'admin3Name'})

pcode_col = next((c for c in adm3_aoi.columns if 'pcode' in c.lower() and '3' in c), None)
keep_cols = ['admin3Name', 'geometry'] + ([pcode_col] if pcode_col else [])
adm3_aoi = adm3_aoi[[c for c in keep_cols if c in adm3_aoi.columns]]
print(adm3_aoi[['admin3Name']].head(10).to_string())


  Loaded ADM3 from cache: data\raw\boundaries\drc_adm3.gpkg
  ADM3 units in AOI: 23
  Name column: shapeName
    admin3Name
8       Bukavu
10     Butembo
11        Beni
83       Irumu
84     Mambasa
85       Djugu
86      Mahagi
87         Aru
88  Nyiragongo
89    Walikale


## 3  H3-7 hexagon grid

In [9]:
# Generate H3-7 hexagons covering the AOI bounding box
aoi_poly = {
    'type': 'Polygon',
    'coordinates': [[
        [AOI_BBOX['min_lon'], AOI_BBOX['min_lat']],
        [AOI_BBOX['max_lon'], AOI_BBOX['min_lat']],
        [AOI_BBOX['max_lon'], AOI_BBOX['max_lat']],
        [AOI_BBOX['min_lon'], AOI_BBOX['max_lat']],
        [AOI_BBOX['min_lon'], AOI_BBOX['min_lat']],
    ]]
}

# h3 v4 API: geo_to_cells replaces polyfill_geojson
hex_ids = list(h3.geo_to_cells(aoi_poly, H3_RES))
print(f'H3-7 hexagons in bounding box: {len(hex_ids)}')

# Clip to provinces union
def h3_to_polygon(h):
    # h3 v4: cell_to_boundary returns (lat, lng) tuples
    coords = [(lng, lat) for lat, lng in h3.cell_to_boundary(h)]
    return Polygon(coords)

hex_gdf = gpd.GeoDataFrame(
    {'h3_index': hex_ids},
    geometry=[h3_to_polygon(h) for h in hex_ids],
    crs='EPSG:4326'
)
hex_gdf = hex_gdf[hex_gdf.geometry.intersects(prov_union)].copy()
print(f'H3-7 hexagons clipped to provinces: {len(hex_gdf)}')

H3-7 hexagons in bounding box: 75564


H3-7 hexagons clipped to provinces: 31358


## 4  Spatial join: flood extent → admin-2 (territory)


In [10]:
admin2_rows = []

for month, gdf in tqdm(flood_gdfs.items(), desc='Admin-2 join'):
    if gdf.empty:
        continue
    gdf = gdf.to_crs('EPSG:4326')
    joined = gpd.overlay(adm2_aoi[['shapeName', 'shapeISO', 'geometry']], gdf, how='intersection')
    if joined.empty:
        continue
    joined = joined.to_crs('EPSG:32735')
    joined['flood_area_km2'] = joined.geometry.area / 1e6
    for _, row in adm2_aoi.iterrows():
        terr_flood = joined[joined['shapeName'] == row['shapeName']]['flood_area_km2'].sum()
        admin2_rows.append({
            'month': month,
            'shapeName': row['shapeName'],
            'shapeISO': row.get('shapeISO', ''),
            'flood_area_km2': round(terr_flood, 3),
            'quality': df.loc[month, 'quality'] if month in df.index else 'valid',
        })

admin2_df  = pd.DataFrame(admin2_rows)
admin2_gdf = adm2_aoi[['shapeName', 'shapeISO', 'geometry']].merge(
    admin2_df, on=['shapeName', 'shapeISO'], how='left')
print(f'Admin-2 rows: {len(admin2_df)}')
print(admin2_df[admin2_df['flood_area_km2'] > 0].head(10).to_string())


Admin-2 join:   0%|          | 0/16 [00:00<?, ?it/s]

Admin-2 join:  56%|█████▋    | 9/16 [00:00<00:00, 80.69it/s]

Admin-2 join: 100%|██████████| 16/16 [00:00<00:00, 68.77it/s]

Admin-2 rows: 261
       month shapeName shapeISO  flood_area_km2 quality
12   2025-05     Irumu                    0.100   valid
18   2025-05   Mambasa                    4.120   valid
24   2025-05  Rutshuru                    1.780   valid
41   2025-06     Irumu                    4.280   valid
74   2025-09    Lubero                    0.071   valid
77   2025-09    Masisi                    0.030   valid
82   2025-09  Rutshuru                    3.119   valid
84   2025-09     Uvira                    0.020   valid
99   2025-12     Irumu                    0.180   valid
128  2026-01     Irumu                    0.350   valid


## 4b  Spatial join: flood extent → admin-3 (OCHA HDX secteur/chefferie)


In [11]:
admin3_rows = []

for month, gdf in tqdm(flood_gdfs.items(), desc='Admin-3 join'):
    if gdf.empty:
        continue
    gdf = gdf.to_crs('EPSG:4326')
    joined = gpd.overlay(adm3_aoi[['admin3Name', 'geometry']], gdf, how='intersection')
    if joined.empty:
        continue
    joined = joined.to_crs('EPSG:32735')
    joined['flood_area_km2'] = joined.geometry.area / 1e6
    for _, row in adm3_aoi.iterrows():
        unit_flood = joined[joined['admin3Name'] == row['admin3Name']]['flood_area_km2'].sum()
        admin3_rows.append({
            'month': month,
            'admin3Name': row['admin3Name'],
            'flood_area_km2': round(unit_flood, 3),
            'quality': df.loc[month, 'quality'] if month in df.index else 'valid',
        })

admin3_df  = pd.DataFrame(admin3_rows)
admin3_gdf = adm3_aoi[['admin3Name', 'geometry']].merge(admin3_df, on='admin3Name', how='left')
print(f'Admin-3 rows: {len(admin3_df)}')
print(admin3_df[admin3_df['flood_area_km2'] > 0].head(10).to_string())


Admin-3 join:   0%|          | 0/16 [00:00<?, ?it/s]

Admin-3 join:  44%|████▍     | 7/16 [00:00<00:00, 69.23it/s]

Admin-3 join:  94%|█████████▍| 15/16 [00:00<00:00, 68.96it/s]

Admin-3 join: 100%|██████████| 16/16 [00:00<00:00, 60.81it/s]

Admin-3 rows: 207
       month admin3Name  flood_area_km2 quality
3    2025-05      Irumu           0.100   valid
4    2025-05    Mambasa           4.120   valid
12   2025-05   Rutshuru           1.780   valid
26   2025-06      Irumu           0.713   valid
58   2025-09   Rutshuru           3.119   valid
59   2025-09     Masisi           0.030   valid
61   2025-09      Uvira           0.020   valid
72   2025-12      Irumu           0.180   valid
95   2026-01      Irumu           0.135   valid
119  2026-02    Mambasa           3.520   valid


## 5  Spatial join: flood extent → H3-7

In [12]:
h3_rows = []

for month, gdf in tqdm(flood_gdfs.items(), desc='H3-7 join'):
    if gdf.empty:
        continue
    gdf = gdf.to_crs('EPSG:4326')
    joined = gpd.overlay(hex_gdf[['h3_index', 'geometry']], gdf, how='intersection')
    if joined.empty:
        continue
    joined = joined.to_crs('EPSG:32735')
    joined['flood_area_km2'] = joined.geometry.area / 1e6
    flood_by_hex = joined.groupby('h3_index')['flood_area_km2'].sum().reset_index()
    flood_by_hex['month'] = month
    flood_by_hex['quality'] = df.loc[month, 'quality'] if month in df.index else 'valid'
    h3_rows.append(flood_by_hex)

h3_df = pd.concat(h3_rows, ignore_index=True) if h3_rows else pd.DataFrame()
h3_gdf = hex_gdf.merge(h3_df, on='h3_index', how='left')
print(f'H3-7 rows: {len(h3_df)}')
print(h3_df[h3_df['flood_area_km2'] > 0].head(10).to_string())

H3-7 join:   0%|          | 0/16 [00:00<?, ?it/s]

H3-7 join:  12%|█▎        | 2/16 [00:00<00:01, 13.81it/s]

H3-7 join:  31%|███▏      | 5/16 [00:00<00:00, 17.85it/s]

H3-7 join:  44%|████▍     | 7/16 [00:05<00:10,  1.13s/it]

H3-7 join:  62%|██████▎   | 10/16 [00:06<00:03,  1.57it/s]

H3-7 join:  75%|███████▌  | 12/16 [00:06<00:01,  2.01it/s]

H3-7 join:  94%|█████████▍| 15/16 [00:06<00:00,  2.80it/s]

H3-7 join: 100%|██████████| 16/16 [00:08<00:00,  1.67it/s]

H3-7 join: 100%|██████████| 16/16 [00:08<00:00,  1.84it/s]

H3-7 rows: 298
          h3_index  flood_area_km2    month quality
0  876ac0a70ffffff        0.060108  2025-05   valid
1  876ac0a76ffffff        0.039892  2025-05   valid
2  876ac4511ffffff        0.280000  2025-05   valid
3  876ac4528ffffff        0.637100  2025-05   valid
4  876ac4529ffffff        2.341363  2025-05   valid
5  876ac452dffffff        0.848902  2025-05   valid
6  876ac4cdaffffff        0.012635  2025-05   valid
7  876adc910ffffff        1.613769  2025-05   valid
8  876adc912ffffff        0.002973  2025-05   valid
9  876adc916ffffff        0.163258  2025-05   valid


## 6  Cell tower locations (OpenCelliD)

Register at [opencellid.org](https://opencellid.org) to get a free API token.  
Set `OPENCELLID_TOKEN` below and re-run this cell to download DRC towers (MCC=630).

In [13]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(__file__).parent.parent / '.env' if '__file__' in dir() else Path('..') / '.env')
OPENCELLID_TOKEN = os.getenv('OPENCELLID_TOKEN', '')  # set in .env

towers_gdf = None

if OPENCELLID_TOKEN:
    import io
    tower_cache = Path('data/raw/opencellid_cod.csv')
    if not tower_cache.exists():
        print('Downloading DRC cell towers from OpenCelliD...')
        url = f'https://opencellid.org/ocid/downloads?token={OPENCELLID_TOKEN}&type=mcc&file=630.csv.gz'
        r = requests.get(url, stream=True, timeout=120)
        tower_cache.parent.mkdir(parents=True, exist_ok=True)
        with open(tower_cache.with_suffix('.csv.gz'), 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        import gzip, shutil
        with gzip.open(tower_cache.with_suffix('.csv.gz'), 'rb') as f_in:
            with open(tower_cache, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f'  Saved to {tower_cache}')

    towers_raw = pd.read_csv(tower_cache,
        names=['radio','mcc','net','area','cell','unit','lon','lat','range','samples',
               'changeable','created','updated','averageSignal'])
    towers_raw = towers_raw[(towers_raw.lat.between(AOI_BBOX['min_lat'], AOI_BBOX['max_lat'])) &
                            (towers_raw.lon.between(AOI_BBOX['min_lon'], AOI_BBOX['max_lon']))]
    towers_gdf = gpd.GeoDataFrame(towers_raw,
        geometry=gpd.points_from_xy(towers_raw.lon, towers_raw.lat), crs='EPSG:4326')
    print(f'  Towers in AOI: {len(towers_gdf)}')

    # Join tower counts to admin-2 territories
    t_adm2 = gpd.sjoin(towers_gdf[['radio','geometry']], adm2_aoi[['shapeName','geometry']], how='left', predicate='within')
    adm2_tower_counts = t_adm2.groupby('shapeName').size().reset_index(name='tower_count')
    admin2_gdf = admin2_gdf.merge(adm2_tower_counts, on='shapeName', how='left')
    admin2_gdf['tower_count'] = admin2_gdf['tower_count'].fillna(0).astype(int)

    # Join tower counts to admin-3 secteurs
    t_adm3 = gpd.sjoin(towers_gdf[['radio','geometry']], adm3_aoi[['admin3Name','geometry']], how='left', predicate='within')
    adm3_tower_counts = t_adm3.groupby('admin3Name').size().reset_index(name='tower_count')
    admin3_gdf = admin3_gdf.merge(adm3_tower_counts, on='admin3Name', how='left')
    admin3_gdf['tower_count'] = admin3_gdf['tower_count'].fillna(0).astype(int)

    # Join tower counts to H3-7
    t_hex = gpd.sjoin(towers_gdf[['radio','geometry']], hex_gdf[['h3_index','geometry']], how='left', predicate='within')
    hex_tower = t_hex.groupby('h3_index').size().reset_index(name='tower_count')
    h3_gdf = h3_gdf.merge(hex_tower, on='h3_index', how='left')
    h3_gdf['tower_count'] = h3_gdf['tower_count'].fillna(0).astype(int)
    print('Tower counts joined to admin-2, admin-3, and H3-7.')
else:
    print('No OpenCelliD token set — skipping tower data. Set OPENCELLID_TOKEN in .env and re-run.')


No OpenCelliD token set — skipping tower data. Set OPENCELLID_TOKEN in .env and re-run.


## 7  Export GeoParquet

In [14]:
CSV_DIR = PROJECT_ROOT / 'data' / 'handover' / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)

# ── Admin-2 (World Bank geoBoundaries territories) ─────────────
admin2_out = FRAMES_DIR / 'admin2.parquet'
admin2_gdf.to_parquet(str(admin2_out), index=False)
print(f'Exported admin2.parquet  ({len(admin2_gdf):,} rows) → {admin2_out}')

admin2_df.to_csv(CSV_DIR / 'admin2_flood.csv', index=False)
admin2_wide = admin2_df.pivot_table(
    index=['shapeName', 'shapeISO'], columns='month', values='flood_area_km2').reset_index()
admin2_wide.columns.name = None
admin2_wide.to_csv(CSV_DIR / 'admin2_flood_wide.csv', index=False)
print(f'  Exported admin2_flood.csv + admin2_flood_wide.csv → {CSV_DIR}')

# ── Admin-3 (OCHA HDX secteur/chefferie) ───────────────────────
admin3_out = FRAMES_DIR / 'admin3.parquet'
admin3_gdf.to_parquet(str(admin3_out), index=False)
print(f'Exported admin3.parquet  ({len(admin3_gdf):,} rows) → {admin3_out}')

admin3_df.to_csv(CSV_DIR / 'admin3_flood.csv', index=False)
admin3_wide = admin3_df.pivot_table(
    index='admin3Name', columns='month', values='flood_area_km2').reset_index()
admin3_wide.columns.name = None
admin3_wide.to_csv(CSV_DIR / 'admin3_flood_wide.csv', index=False)
print(f'  Exported admin3_flood.csv + admin3_flood_wide.csv → {CSV_DIR}')

# ── H3-7 ────────────────────────────────────────────────────────
h3_out = FRAMES_DIR / 'h3_7.parquet'
h3_gdf.to_parquet(str(h3_out), index=False)
print(f'Exported h3_7.parquet    ({len(h3_gdf):,} rows)  → {h3_out}')


Exported admin2.parquet  (261 rows) → data\outputs\sampling_frames\admin2.parquet
  Exported admin2_flood.csv + admin2_flood_wide.csv → C:\Users\trevm\Projects\Floodmaps\data\handover\csv


Exported admin3.parquet  (207 rows) → data\outputs\sampling_frames\admin3.parquet
  Exported admin3_flood.csv + admin3_flood_wide.csv → C:\Users\trevm\Projects\Floodmaps\data\handover\csv


Exported h3_7.parquet    (31,374 rows)  → data\outputs\sampling_frames\h3_7.parquet


## 8  Summary

In [15]:
print('=== Sampling Frame Summary ===')
print(f'Admin-2 territories (World Bank) : {adm2_aoi.shapeName.nunique()}')
print(f'Admin-3 units (OCHA HDX)         : {adm3_aoi.admin3Name.nunique()}')
print(f'H3-7 hexagons in AOI             : {hex_gdf.h3_index.nunique()}')
print(f'Valid flood months               : {len(flood_gdfs)}')
if towers_gdf is not None:
    print(f'Cell towers in AOI               : {len(towers_gdf)}')
else:
    print('Cell towers                      : not loaded (no token)')
print()
print('Top 10 territories by peak flood area (Admin-2):')
peak2 = admin2_df.groupby('shapeName')['flood_area_km2'].max().sort_values(ascending=False).head(10)
print(peak2.to_string())
print()
print('Top 10 units by peak flood area (Admin-3):')
peak3 = admin3_df.groupby('admin3Name')['flood_area_km2'].max().sort_values(ascending=False).head(10)
print(peak3.to_string())


=== Sampling Frame Summary ===
Admin-2 territories (World Bank) : 29
Admin-3 units (OCHA HDX)         : 23
H3-7 hexagons in AOI             : 31358
Valid flood months               : 16
Cell towers                      : not loaded (no token)

Top 10 territories by peak flood area (Admin-2):
shapeName
Irumu       10.270
Uvira        9.142
Kabare       5.768
Mambasa      4.120
Bukavu       3.509
Rutshuru     3.119
Fizi         2.790
Masisi       0.670
Walungu      0.468
Kalehe       0.342

Top 10 units by peak flood area (Admin-3):
admin3Name
Irumu       10.270
Uvira        8.822
Kabare       5.334
Mambasa      4.120
Bukavu       3.281
Rutshuru     3.119
Fizi         1.852
Masisi       0.670
Walungu      0.468
Kalehe       0.317


## 9  Interactive map — flood extents + admin-3 boundaries + cell towers

In [16]:
import folium, math
from folium.plugins import MarkerCluster

m = folium.Map(location=[-1.5, 28.8], zoom_start=7, tiles='CartoDB positron')
prov_gdf = gpd.GeoDataFrame(geometry=[prov_union], crs='EPSG:4326')

def adm2_color(peak):
    if peak > 100: return '#d73027'
    if peak > 10:  return '#fc8d59'
    if peak > 1:   return '#fee090'
    return '#e0f3f8'

def adm3_color(peak):
    """Secteur scale — lower thresholds than territory level."""
    if peak > 20:  return '#d73027'
    if peak > 5:   return '#fc8d59'
    if peak > 0.5: return '#fee090'
    return '#e0f3f8'

def hex_color(peak):
    """H3-7 scale (~5 km2 cells) — blue ramp."""
    if peak > 2:   return '#08306b'
    if peak > 0.5: return '#2171b5'
    if peak > 0:   return '#9ecae1'
    return None  # empty hex — skip

# --- Admin-2: territories ---
admin2_layer = folium.FeatureGroup(name='Admin-2 territories (World Bank)', show=True)
for _, row in adm2_aoi.iterrows():
    peak_vals = admin2_df[admin2_df['shapeName'] == row['shapeName']]['flood_area_km2']
    peak = float(peak_vals.max()) if len(peak_vals) else 0.0
    if math.isnan(peak): peak = 0.0
    color = adm2_color(peak)
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, c=color: {'fillColor': c, 'color': '#333', 'weight': 1.5, 'fillOpacity': 0.55},
        tooltip=f"{row['shapeName']}  (peak: {peak:,.1f} km2)"
    ).add_to(admin2_layer)
admin2_layer.add_to(m)

# --- Admin-3: secteurs/chefferies ---
admin3_layer = folium.FeatureGroup(name='Admin-3 secteurs (OCHA HDX)', show=True)
for _, row in adm3_aoi.iterrows():
    peak_vals = admin3_df[admin3_df['admin3Name'] == row['admin3Name']]['flood_area_km2']
    peak = float(peak_vals.max()) if len(peak_vals) else 0.0
    if math.isnan(peak): peak = 0.0
    color = adm3_color(peak)
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, c=color: {'fillColor': c, 'color': '#444', 'weight': 0.8, 'fillOpacity': 0.6},
        tooltip=f"{row['admin3Name']}  (peak: {peak:,.1f} km2)"
    ).add_to(admin3_layer)
admin3_layer.add_to(m)

# --- H3-7: only flooded hexes, skip empty ones for performance ---
hex_peak = h3_gdf.groupby('h3_index')['flood_area_km2'].max() if 'flood_area_km2' in h3_gdf.columns else {}
hex_layer = folium.FeatureGroup(name='H3-7 flood hexes', show=False)
flooded_count = 0
for _, row in hex_gdf.iterrows():
    peak = float(hex_peak.get(row['h3_index'], 0.0)) if hasattr(hex_peak, 'get') else 0.0
    color = hex_color(peak)
    if color is None:
        continue
    flooded_count += 1
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, c=color: {'fillColor': c, 'color': c, 'weight': 0.3, 'fillOpacity': 0.65}
    ).add_to(hex_layer)
hex_layer.add_to(m)
print(f'Flooded hexes added to map: {flooded_count}')

# --- Flood extents ---
for month, gdf in flood_gdfs.items():
    gdf_clipped = gpd.overlay(gdf.to_crs('EPSG:4326'), prov_gdf, how='intersection')
    if gdf_clipped.empty:
        continue
    fg = folium.FeatureGroup(name=f'Flood {month}', show=(month == '2025-09'))
    folium.GeoJson(
        gdf_clipped.__geo_interface__,
        style_function=lambda _: {'fillColor': '#08519c', 'color': '#08306b', 'weight': 0.4, 'fillOpacity': 0.6}
    ).add_to(fg)
    fg.add_to(m)

# --- Cell towers ---
if towers_gdf is not None and len(towers_gdf) > 0:
    tower_layer = folium.FeatureGroup(name='Cell towers', show=False)
    cluster = MarkerCluster().add_to(tower_layer)
    for _, t in towers_gdf.iterrows():
        folium.CircleMarker(
            location=[t.lat, t.lon], radius=3,
            color='#e31a1c', fill=True, fill_opacity=0.7,
            tooltip=f"{t.get('radio', '')} net:{t.get('net', '')}"
        ).add_to(cluster)
    tower_layer.add_to(m)
else:
    print('No tower data — set OPENCELLID_TOKEN in .env and re-run.')

# --- AOI outline ---
aoi_coords = [[-5.9, 26.8], [-5.9, 30.8], [3.0, 30.8], [3.0, 26.8], [-5.9, 26.8]]
folium.PolyLine(aoi_coords, color='#555', weight=1.5, dash_array='6 4', tooltip='AOI: Eastern DRC').add_to(m)

legend_html = """
<div style='position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
            padding:12px;border-radius:6px;border:1px solid #ccc;font-size:12px;line-height:1.8'>
  <b>Admin-2 peak flood</b><br>
  <span style='background:#d73027;padding:2px 10px'>&nbsp;</span> &gt;100 km&#178;<br>
  <span style='background:#fc8d59;padding:2px 10px'>&nbsp;</span> 10&ndash;100 km&#178;<br>
  <span style='background:#fee090;padding:2px 10px'>&nbsp;</span> 1&ndash;10 km&#178;<br>
  <span style='background:#e0f3f8;padding:2px 10px'>&nbsp;</span> &lt;1 km&#178;<br>
  <b>Admin-3 peak flood</b><br>
  <span style='background:#d73027;padding:2px 10px'>&nbsp;</span> &gt;20 km&#178;<br>
  <span style='background:#fc8d59;padding:2px 10px'>&nbsp;</span> 5&ndash;20 km&#178;<br>
  <span style='background:#fee090;padding:2px 10px'>&nbsp;</span> 0.5&ndash;5 km&#178;<br>
  <span style='background:#e0f3f8;padding:2px 10px'>&nbsp;</span> &lt;0.5 km&#178;<br>
  <b>H3-7 hex flood</b><br>
  <span style='background:#08306b;padding:2px 10px'>&nbsp;</span> &gt;2 km&#178;<br>
  <span style='background:#2171b5;padding:2px 10px'>&nbsp;</span> 0.5&ndash;2 km&#178;<br>
  <span style='background:#9ecae1;padding:2px 10px'>&nbsp;</span> trace
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

map_path = Path('docs/flood_sampling_map.html')
map_path.parent.mkdir(exist_ok=True)
m.save(str(map_path))
print(f'Map saved: {map_path}')
import webbrowser
webbrowser.open(map_path.resolve().as_uri())
m


Flooded hexes added to map: 282


No tower data — set OPENCELLID_TOKEN in .env and re-run.


Map saved: docs\flood_sampling_map.html
